# QuickCart Support Bot — run in Google Colab

A customer-support chat app (**React** frontend + **Flask** backend + **OpenAI**).
Run the cells top to bottom (**Runtime → Run all**). At the end you get a **public link** you can open or share.

**Before you run:** add your OpenAI key in Colab → 🔑 **Secrets** (left sidebar) as `OPENAI_API_KEY` and switch *Notebook access* on.
If you skip this, the notebook will ask you to paste the key.

### 1. Get the code

In [ ]:
import os, subprocess

REPO = "https://github.com/Venkat4655/ai-video-m1demo"
APP_DIR = "/content/ai-video-m1demo/Prompt-Example"

if not os.path.isdir(APP_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/ai-video-m1demo"], check=True)
print("Code ready at", APP_DIR)

### 2. Install Python and Node dependencies, build the React app

In [ ]:
%%bash
set -e
cd /content/ai-video-m1demo/Prompt-Example

pip install -q -r backend/requirements.txt

# Vite needs a recent Node. Install Node 22 if the Colab image has an older one.
NODE_MAJOR=$(node -v 2>/dev/null | sed 's/v\([0-9]*\).*/\1/' || echo 0)
if [ "${NODE_MAJOR:-0}" -lt 22 ]; then
  curl -fsSL https://deb.nodesource.com/setup_22.x | bash - >/dev/null 2>&1
  apt-get install -y -qq nodejs >/dev/null 2>&1
fi
node -v

cd frontend
npm install --no-audit --no-fund --loglevel=error
npm run build 2>&1 | tail -5

### 3. Set your OpenAI API key

In [ ]:
import os, getpass

key = None
try:
    from google.colab import userdata
    key = userdata.get("OPENAI_API_KEY")
except Exception:
    pass

if not key:
    key = getpass.getpass("Paste your OpenAI API key (hidden): ").strip()

assert key, "No API key provided."
os.environ["OPENAI_API_KEY"] = key
print("API key loaded.")

### 4. Start the app and create the public link

In [ ]:
import re, subprocess, time, urllib.request, os
from IPython.display import HTML, display, clear_output

PORT = 5050

# --- Flask (also serves the built React app) ---
def start_flask():
    log = open("/content/flask.log", "w")
    return subprocess.Popen(["python", "app.py"], cwd=f"{APP_DIR}/backend",
                            env=dict(os.environ, PORT=str(PORT)),
                            stdout=log, stderr=subprocess.STDOUT)

def flask_up():
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=3)
        return True
    except Exception:
        return False

flask = start_flask()
for _ in range(30):
    if flask_up():
        break
    time.sleep(1)
else:
    raise RuntimeError("Flask did not start:\n" + open("/content/flask.log").read())

# --- Cloudflare quick tunnel: free, no account, public https://*.trycloudflare.com URL ---
if not os.path.exists("/content/cloudflared"):
    subprocess.run(["wget", "-q", "-O", "/content/cloudflared",
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
                   check=True)
    os.chmod("/content/cloudflared", 0o755)

def start_tunnel():
    log = open("/content/cloudflared.log", "w")
    proc = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}",
                             "--no-autoupdate", "--protocol", "http2"],
                            stdout=log, stderr=subprocess.STDOUT)
    # Wait until the tunnel is registered AND the public URL really answers.
    for _ in range(90):
        text = open("/content/cloudflared.log").read()
        urls = [u for u in re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
                if u != "https://api.trycloudflare.com"]  # api.* is Cloudflare's own endpoint
        if urls and "Registered tunnel connection" in text:
            try:
                req = urllib.request.Request(urls[0], headers={"User-Agent": "Mozilla/5.0"})
                if urllib.request.urlopen(req, timeout=5).status == 200:
                    return proc, urls[0]
            except Exception:
                pass  # DNS for a brand-new tunnel can take a few seconds
        if proc.poll() is not None:
            break
        time.sleep(1)
    proc.terminate()
    raise RuntimeError("Could not create a working public link:\n" + open("/content/cloudflared.log").read()[-2000:])

def show(url):
    clear_output(wait=True)
    display(HTML(f'<h3>🌐 Public link: <a href="{url}" target="_blank">{url}</a></h3>'
                 '<p><b>Keep this cell running.</b> Stopping it, or closing/idling Colab, takes the link offline '
                 '(Cloudflare then shows Error 1033). Anyone with the link can use the bot and spend your OpenAI credit.</p>'))

tunnel, public_url = start_tunnel()
show(public_url)

# Keep the cell alive and self-heal: if Flask or the tunnel dies, restart it (a restart gives a NEW link).
try:
    while True:
        time.sleep(20)
        if flask.poll() is not None or not flask_up():
            flask = start_flask(); time.sleep(3)
        if tunnel.poll() is not None:
            tunnel, public_url = start_tunnel()
            show(public_url)
except KeyboardInterrupt:
    flask.terminate(); tunnel.terminate()
    print("Stopped. The public link is now offline.")

### Try it
| Message | What happens |
| --- | --- |
| `Where is QC-1001?` | Looks up the order and answers from its details |
| `What is your return policy?` | Answers from `faq.txt` |
| `Where is my order?` | The bot asks for the order number |
| `Where is QC-9999?` | The app replies directly: order not found (no LLM call) |

Open **Show Under the Hood** in the app to see the Rules, Facts and Question sent to the model.

To stop everything, press the ■ stop button on the cell above.